# 1.0 Introduction
In this notebook, we will process a weather dataset by performing data cleaning and analysis.



source: https://www.kaggle.com/datasets/shahmirvarqha/weather-data-malaysia/data?select=full_weather.csv

| Column Name            | Description                                                                 | Units                |
|------------------------|-----------------------------------------------------------------------------|----------------------|
| `datetime`             | The recorded date and time of the weather observation.                      |                      |
| `place`                | Specific location where the weather data was collected.                     |                      |
| `city`                 | The city corresponding to the recorded weather data.                        |                      |
| `state`                | The state in Malaysia where the observation was made.                       |                      |
| `temperature`          | Air temperature measured in degrees Celsius.                                | °C                   |
| `pressure`             | Atmospheric pressure measured in hectopascals.                              | hPa                  |
| `dew_point`            | The temperature at which air becomes saturated.                             | °C                   |
| `humidity`             | The percentage of moisture in the air.                                      | %                    |
| `wind_speed`           | Speed of the wind measured in meters per second.                            | m/s                  |
| `gust`                 | Sudden bursts of wind speed.                                                | m/s                  |
| `wind_chill`           | The perceived temperature due to wind.                                      | °C                   |
| `uv_index`             | A scale indicating the intensity of ultraviolet radiation.                  |                      |
| `feels_like_temperature` | The apparent temperature considering humidity and wind.                     | °C                   |
| `visibility`           | Distance one can see clearly.                                               | km                   |
| `solar_radiation`      | Amount of solar energy received.                                            | W/m²                 |
| `pollutant_value`      | Air pollution concentration.                                                | µg/m³                |
| `precipitation_rate`   | The rate of rainfall.                                                       | mm/h                 |
| `precipitation_total`  | Total accumulated rainfall.                                                 | mm                   |

# 2.0 Importing the Required Libraries

In [ ]:
# Install dependencies as needed:
%pip install kagglehub[pandas-datasets]
%pip install kaggle
%pip install -q kaggle
%pip install pykalman

import kagglehub
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from kagglehub import KaggleDatasetAdapter
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from pykalman import KalmanFilter

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.4/249.4 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.5/149.5 kB 16.7 MB/s eta 0:00:00


In [ ]:
from google.colab import files
files.upload()  # Upload kaggle.json here (you get it from https://www.kaggle.com/settings)

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

TypeError: 'NoneType' object is not subscriptable

In [ ]:
!kaggle datasets files -d shahmirvarqha/weather-data-malaysia

In [ ]:
!kaggle datasets download -d shahmirvarqha/weather-data-malaysia -p /content/weather --unzip

In [ ]:
print(os.listdir("/content/weather"))

# Load the main weather data CSV
df = pd.read_csv("/content/weather/full_weather.csv")

In [ ]:
df.head()

In [ ]:
# Check the size of the dataset
print(f"Size of the original dataset: {df.shape}")

# 3.0 Check on available cities and places

In [ ]:
df = df[(df["datetime"] >= '2011-01-01 00:00:00') & (df["datetime"] <= '2023-12-31 23:59:59')]

In [ ]:
# Filter years from 2011 to 2023
print(f"Size of the dataset after filter: {df.shape}")

In [ ]:
print(df['city'].unique())

In [ ]:
print(df['place'].unique())

In [ ]:
print(df['state'].unique())

In [ ]:
# # Get the list of unique places
# target_places = df['place'].unique()
# # print the place, city and state of each targeted places
# # Iterate through the target places and print city, place, and state
# for place in target_places:
#   # Find the first row for this place to get the city and state
#   row_df = df[df['place'] == place]
#   if not row_df.empty:
#     row = row_df.iloc[0]
#     print(f"Place: {row['place']}, City: {row['city']}, State: {row['state']}")
#   else:
#     print(f"Place: {place} not found in the filtered dataset.")

## 3.1 Check on data availability

Based on the Information and communication technology for fisheries industry development in Malaysia, we will only pick the below for better weather accuracy:

reference: https://www.researchgate.net/publication/267771406_Information_and_communication_technology_for_fisheries_industry_development_in_Malaysia

1. Perlis - Padang Besar
2. Kedah - Kampung Paya Mempelam
3. Penang - Batu Maung
4. Perak - Pekan Sitiawan
5. Selangor - Taman Setia
6. Negeri Sembilan - Nilai Impian
7. Melaka - Durian Tunggal
8. Johor - Mersing
9. Pahang - Pekan
10. Terengganu - Kuala Nerus
11. Kelantan - Kg Leban Chondong
12. Sarawak - Kota Sentosa
13. Sabah - Tanjung Aru
14. Labuan - Bandar Victoria

In [ ]:
# Define the target places
target_places = [
    # Perlis
    "Kuala Perlis",
    # Kedah
    "Kampung Paya Mempelam",
    # Penang
    "Batu Maung",
    # Sitiawan
    "Pekan Sitiawan",
    # Selangor
    "Taman Eng Ann",
    # Negeri Sembilan
    "Nilai Impian",
    # Melaka
    "Durian Tunggal",
    # Johor
    "Mersing",
    # Pahang
    "Pekan",
    # Terengganu
    "Kuala Nerus",
    # Kelantam
    "Sabak",
    # Sarawak
    "Kota Sentosa",
    # Sabah
    "Tanjung Aru",
    # Labuan
    "Bandar Victoria",
]

# Filter rows where `place` is in the list
df_filtered = df[df["place"].isin(target_places)]

In [ ]:
# print the place, city and state of each targeted places

# Iterate through the target places and print city, place, and state
for place in target_places:
  # Find the first row for this place to get the city and state
  row_df = df_filtered[df_filtered['place'] == place]
  if not row_df.empty:
    row = row_df.iloc[0]
    print(f"Place: {row['place']}, City: {row['city']}, State: {row['state']}")
  else:
    print(f"Place: {place} not found in the filtered dataset.")

In [ ]:
# Result
df_filtered.head()

In [ ]:
#Understanding the size of the dataset after filtering it
print(f"Size of the filtered dataset: {df_filtered.shape}")

# 4.0 Data Preprocessing

In [ ]:
df_filtered_copy = df_filtered.copy()

##  4.1 Identifying and Handling Duplicate Data

In [ ]:
# check number of duplicated rows
df_filtered_copy.duplicated().sum()

Result: No duplicates found from the filtered dataset

## 4.2 Identifying and Handling Missing Values

In [ ]:
# function that check for missing data
def missing_statistics(df):
    statitics = pd.DataFrame(df.isnull().sum()).reset_index()
    statitics.columns=['COLUMN NAME',"MISSING VALUES"]
    statitics['TOTAL ROWS'] = df.shape[0]
    statitics['% MISSING'] = round((statitics['MISSING VALUES']/statitics['TOTAL ROWS'])*100,2)
    return statitics

In [ ]:
missing_statistics(df_filtered_copy)

- Drop columns that has more than 50% missing data
- Perform data imputation on columnns that has less than 50% missing data

### 4.2.1 Drop columns that has more than 45% missing data
*   gust
*   solar_radiation
*   pollutant_value
*   feels_like_temperature
*   visibility
*   precipitation_rate
*   precipitation_total



In [ ]:
columns_to_drop = [
    "gust",
    "solar_radiation",
    "pollutant_value",
    "feels_like_temperature",
    "visibility",
    "precipitation_rate",
    "precipitation_total"
]
df_filtered_copy = df_filtered_copy.drop(columns=columns_to_drop)

In [ ]:
df_filtered_copy.head()

### 4.2.2 Perform interpolation data imputation on columns that has less than 50% missing data

*   temperature
*   pressure
*   dew_point
*   humidity
*   wind_speed
*   wind_chill
*   uv_index




In [ ]:
# Perform kalman fitler on remaining columns with missing values
columns_to_interpolate = [
    "temperature",
    "pressure",
    "dew_point",
    "humidity",
    "wind_speed",
    "wind_chill",
    "uv_index",
]

In [ ]:
# Separate the filtered_df based on unique values in the 'place' column
separated_dfs = {place: df_filtered_copy[df_filtered_copy['place'] == place].copy() for place in df_filtered_copy['place'].unique()}

In [ ]:
# Check the size of the datasets
for place, df_place in separated_dfs.items():
  print(f"Size of dataframe for '{place}': {df_place.shape}")

In [ ]:
# Check the missing values
for place, df_place in separated_dfs.items():
  print(f"Missing values statistics of dataframe for '{place}'")
  print(missing_statistics(df_place))
  print('\n\n')

#### 4.2.2.1 Handle Missing Range of Dataset



Find the average weather data for every month for each state

In [ ]:
# replace NaN values using 0
df_filtered_copy.fillna(0, inplace=True)
df_filtered_copy.replace('NaN', 0)
df_filtered_copy.head()

In [ ]:
# Add Month and Year columns from the datetime
df_filtered['year'] = pd.to_datetime(df_filtered['datetime']).dt.year
df_filtered['month'] = pd.to_datetime(df_filtered['datetime']).dt.month

In [ ]:
# Group by 'month','year' and 'place' and calculate the mean for the desired columns
# Define the desired columns for aggregation
desired_columns = [
    "temperature",
    "pressure",
    "dew_point",
    "humidity",
    "wind_speed",
    "wind_chill",
    "uv_index",
]

# Group by 'month', 'year', and 'place' and calculate the mean for the desired columns
monthly_yearly_place_mean = df_filtered.groupby(['month', 'year', 'state'])[desired_columns].mean().reset_index()

monthly_yearly_place_mean.head()

In [ ]:
print(f'Size of the monthly_yearly_place_mean: {monthly_yearly_place_mean.shape}')

Replace the 0 values using through data imputation throght Kalman Fitler

In [ ]:
# Make a copy to avoid modifying the original
df_kalman_filled = monthly_yearly_place_mean.copy()

# Columns to apply Kalman filtering on (excluding non-numeric metadata columns)
# Adjust this list if some columns are not meant for filtering
numeric_columns = df_kalman_filled.select_dtypes(include=[np.number]).columns.difference(['month', 'year']).tolist()

# Iterate through the desired columns for Kalman filtering
for column in numeric_columns:
    print(f"Applying Kalman filter to column: {column}")

    # Extract the data
    data = df_kalman_filled[column].astype(float).values

    # Replace 0 with NaN if 0 is considered invalid/missing
    data[data == 0] = np.nan

    # Check if all values are NaN
    if np.isnan(data).all():
        print(f"⚠️ Skipping {column}: all values are NaN.")
        continue

    # Temporary fill for Kalman initialization
    filled_series = pd.Series(data).fillna(method='ffill').fillna(method='bfill')

    # Initialize Kalman Filter
    kf = KalmanFilter(
        transition_matrices=[1],
        observation_matrices=[1],
        initial_state_mean=filled_series.iloc[0],
        initial_state_covariance=1,
        observation_covariance=1,
        transition_covariance=0.01
    )

    # Apply Kalman smoothing
    smoothed_state_means, _ = kf.smooth(filled_series)

    # Replace original values with smoothed values
    smoothed_series = pd.Series(smoothed_state_means.flatten())
    smoothed_series[pd.isna(data)] = np.nan  # reinsert original NaNs if needed

    df_kalman_filled[column] = smoothed_series.values

# Check missing data after Kalman filtering
print("\n✅ Missing values after Kalman filtering:")
print(missing_statistics(df_kalman_filled))

In [ ]:
# Preview the filtered DataFrame
df_kalman_filled.head()

Replace NaN values with random values within the range of each column

In [ ]:
# Replace NaN values with random values within the range of each column

# Function to replace NaN with random values within column range
def replace_nan_with_random_in_range(df):
  for col in df.columns:
    if df[col].isnull().any():
      # Calculate the range of non-NaN values in the column
      min_val = df[col].min()
      max_val = df[col].max()

      # Find the indices where values are NaN
      nan_indices = df[df[col].isnull()].index

      # Generate random values within the range for NaN indices
      random_values = np.random.uniform(min_val, max_val, size=len(nan_indices))

      # Replace NaN values with the generated random values
      df.loc[nan_indices, col] = random_values
  return df

# Apply the function to the DataFrame
df_kalman_filled_random = replace_nan_with_random_in_range(df_kalman_filled.copy())

# Check missing data after replacing NaN with random values
print("\n✅ Missing values after replacing NaN with random values in range:")
print(missing_statistics(df_kalman_filled_random))

# Preview the updated DataFrame
df_kalman_filled_random.head()


Print the range of the month and year for each state and write down the missing month and year

In [ ]:
# Step 1: Group by state and collect available year-month tuples
state_year_month_ranges = df_kalman_filled_random.groupby('state').apply(
    lambda x: sorted(list(zip(x['year'], x['month'])))
)

# Step 2: Create a complete list of all expected year-months
all_year_month = sorted([(year, month) for year in range(2011, 2024) for month in range(1, 13)])

# Step 3: Loop through each state and collect missing months
missing_data_dict = {}
for state, year_month_list in state_year_month_ranges.items():
    if not year_month_list:
        continue
    present_set = set(year_month_list)
    missing = sorted(list(set(all_year_month) - present_set))
    # Convert to string format 'YYYY-MM'
    missing_formatted = [f"{year}-{month:02d}" for year, month in missing]
    missing_data_dict[state] = missing_formatted

# Step 4: Function to generate full date range
def generate_full_monthly_range(start, end):
    return pd.date_range(start=start, end=end, freq='MS').strftime('%Y-%m').tolist()

# Step 5: Function to fill missing data
def fill_missing_data(state_df, state_name, missing_months):
    if state_df.empty:
        return state_df

    # Get the date range from the existing data
    min_date = f"{state_df['year'].min()}-{state_df['month'].min():02d}"
    max_date = f"{state_df['year'].max()}-{state_df['month'].max():02d}"
    full_range = generate_full_monthly_range(min_date, max_date)

    # Create a DataFrame with the full date range
    full_df = pd.DataFrame({'date': full_range})
    full_df['state'] = state_name

    # Add missing date columns to original data
    state_df['date'] = pd.to_datetime(state_df['year'].astype(str) + '-' + state_df['month'].astype(str).str.zfill(2))
    state_df['date'] = state_df['date'].dt.strftime('%Y-%m')

    # Merge
    merged = pd.merge(full_df, state_df, on=['state', 'date'], how='left')

    # Interpolate or fill missing values
    merged = merged.sort_values('date')
    merged.interpolate(method='linear', inplace=True)

    return merged

# Step 6: Apply to all states
filled_states = []
for state in df_kalman_filled_random['state'].unique():
    state_df = df_kalman_filled_random[df_kalman_filled_random['state'] == state]
    missing_months = missing_data_dict.get(state, [])
    filled = fill_missing_data(state_df.copy(), state, missing_months)
    filled_states.append(filled)

# Step 7: Combine all
df_filled_all_states = pd.concat(filled_states, ignore_index=True)

# Optional: Extract year and month back
df_filled_all_states['date'] = pd.to_datetime(df_filled_all_states['date'])
df_filled_all_states['year'] = df_filled_all_states['date'].dt.year
df_filled_all_states['month'] = df_filled_all_states['date'].dt.month


In [ ]:
# print the range of the months that available for each state

# Group by state and find the min and max month for each year
state_monthly_range = df_filled_all_states.groupby(['state', 'year'])['month'].agg(['min', 'max'])

# Print the range for each state and year
for state, year_data in state_monthly_range.groupby(level=0):
    print(f"State: {state}")
    for year, month_range in year_data.iterrows():
        min_month = month_range['min']
        max_month = month_range['max']
        print(f"  Year {year}: Month range {min_month} to {max_month}")
    print("-" * 20)

#### 4.2.2.2 Data Imputation
- Negeri Sembilan (2011 - 2020)
- Pahang (2011 - 2018)
- Perlis (2011 - 2021)
- Selangor (2011 - 2015)

In [ ]:
states_to_impute_backwards = {
    'Negeri Sembilan': (2011, 2020),
    'Pahang': (2011, 2018),
    'Perlis': (2011, 2021),
    'Selangor': (2011, 2015),
}

In [ ]:
from itertools import product

# Function to insert missing rows for a state based on desired year range
def insert_missing_year_months(df, state, start_year, end_year):
    # Existing (year, month) for that state
    existing = set(zip(df[df['state'] == state]['year'], df[df['state'] == state]['month']))

    # Full expected range
    full_range = set(product(range(start_year, end_year + 1), range(1, 13)))

    missing = full_range - existing

    # Create missing rows
    missing_rows = [{
        'state': state,
        'year': y,
        'month': m,
        'date': pd.Timestamp(f'{y}-{m:02d}-01')  # assuming monthly data
    } for y, m in missing]

    return pd.DataFrame(missing_rows)

# Build and append all missing rows for each state
missing_rows_all = []

for state_name, (start_year, end_year) in states_to_impute_backwards.items():
    missing_rows = insert_missing_year_months(df_filled_all_states, state_name, start_year, end_year)
    missing_rows_all.append(missing_rows)

# Combine missing rows and add to main DataFrame
df_filled_all_states = pd.concat([df_filled_all_states] + missing_rows_all, ignore_index=True)

# Sort by state and date
df_filled_all_states = df_filled_all_states.sort_values(['state', 'date']).reset_index(drop=True)


In [ ]:
import numpy as np

# Set random seed
np.random.seed(42)

# Columns to impute
cols_to_impute = ['temperature', 'pressure', 'dew_point', 'humidity', 'wind_speed', 'wind_chill', 'uv_index']

# Function to fill missing values within a group (state)
def month_aware_random_impute(group):
    for col in cols_to_impute:
        for i, row in group.iterrows():
            if pd.isna(row[col]):
                month = row['month']

                # Candidates: same month or neighboring months within the same state
                candidates = group[(group['month'].between(month - 1, month + 1)) & (~group[col].isna())][col]

                # Fallback: if not enough data from same/near month, use all non-null values
                if candidates.empty:
                    candidates = group[col].dropna()

                if not candidates.empty:
                    group.at[i, col] = np.random.choice(candidates.values)
    return group

# Apply imputation by state
df_filled_all_states = (
    df_filled_all_states
    .sort_values(['state', 'date'])
    .groupby('state')
    .apply(month_aware_random_impute)
    .reset_index(drop=True)
)

# 5.0 Exploratory Data Analysis (EDA)

In [ ]:
df_cleaned = df_filled_all_states.copy()

## 5.1 Data information

In [ ]:
# Check the data types of the filtered dataset
df_cleaned.info()

In [ ]:
# Descriibe the data for temperature, pressure, dew_point, humidity, wind_speed, wind_chill, uv_index
df_cleaned[['temperature','pressure','dew_point','humidity','wind_speed','wind_chill', 'uv_index']].describe()

## 5.2 Data Visualization

#### Function to plot timeseries by state

In [ ]:
def plot_timeseries_by_state(df, variable, ylabel, title_prefix):
    """
    Generates timeseries plots for a specified variable for each state.

    Args:
        df (pd.DataFrame): The DataFrame containing the weather data.
                           Must have 'state' and a datetime-like 'date' column
                           and the specified variable column.
        variable (str): The name of the column to plot on the y-axis.
        ylabel (str): The label for the y-axis.
        title_prefix (str): The prefix for the plot title (e.g., 'Temperature').
    """
    # Set the style for the plots
    plt.style.use('seaborn-v0_8-whitegrid')
    # Get unique states
    states = df['state'].unique()

    # Create a plot for each state
    for state in states:
        state_df = df[df['state'] == state].copy()
        # Sort by date
        state_df = state_df.sort_values('date')

        plt.figure(figsize=(15, 5))
        plt.plot(state_df['date'], state_df[variable], label=variable.capitalize())
        plt.title(f'{title_prefix} Over Time for {state}')
        plt.xlabel('Date')
        plt.ylabel(ylabel)
        plt.legend()
        plt.grid(True)
        plt.show()

# Example usage:
# Assuming df_cleaned is your DataFrame with 'state', 'date', and 'temperature'
# plot_timeseries_by_state(df_cleaned, 'temperature', 'Temperature (°C)', 'Temperature')
# plot_timeseries_by_state(df_cleaned, 'humidity', 'Humidity (%)', 'Humidity')
# plot_timeseries_by_state(df_cleaned, 'wind_speed', 'Wind Speed (m/s)', 'Wind Speed')

#### Function to visualize the distribution by state

In [ ]:
# Visualize the distribution of variable for each state
def plot_distribution_by_state(df, variable, xlabel, title_prefix):
    """
    Generates distribution plots for a specified variable for each state.

    Args:
        df (pd.DataFrame): The DataFrame containing the weather data.
                           Must have 'state' and the specified variable column.
        variable (str): The name of the column to plot the distribution for.
        xlabel (str): The label for the x-axis.
        title_prefix (str): The prefix for the plot title (e.g., 'Distribution of Temperature').
    """
    # Set the style for the plots
    plt.style.use('seaborn-v0_8-whitegrid')
    # Get unique states
    states = df['state'].unique()
    # Create distribution plot for the variable for each state
    for state in states:
        state_df = df[df['state'] == state].copy()
        plt.figure(figsize=(10, 6))
        sns.histplot(state_df[variable].dropna(), kde=True, bins=30)
        plt.title(f'{title_prefix} for {state}')
        plt.xlabel(xlabel)
        plt.ylabel('Frequency')
        plt.grid(True)
        plt.show()

# Example usage:
# Assuming df_cleaned is your DataFrame with 'state' and 'temperature'
# plot_distribution_by_state(df_cleaned, 'temperature', 'Temperature (°C)', 'Distribution of Temperature')
# plot_distribution_by_state(df_cleaned, 'humidity', 'Humidity (%)', 'Distribution of Humidity')

#### Function to visualize the distribution for each month in each state using box plot

In [ ]:
# Visualize the distribution for each month in each state using box plot
def plot_boxplot_by_month_by_state(df, variable, ylabel, title_prefix):
    """
    Visualizes the distribution of a variable for each month in each state using box plots.

    Args:
        df (pd.DataFrame): The DataFrame containing the data.
                           Must have 'state', 'month', and the specified variable column.
        variable (str): The name of the column to plot on the y-axis.
        ylabel (str): The label for the y-axis.
        title_prefix (str): The prefix for the plot title (e.g., 'Distribution of Temperature').
    """
    # Set the style for the plots
    plt.style.use('seaborn-v0_8-whitegrid')
    # Get unique states
    states = df['state'].unique()
    # Create box plot for variable distribution by month for each state
    for state in states:
        state_df = df[df['state'] == state].copy()
        plt.figure(figsize=(12, 6))
        sns.boxplot(x='month', y=variable, data=state_df)
        plt.title(f'{title_prefix} by Month for {state}')
        plt.xlabel('Month')
        plt.ylabel(ylabel)
        plt.xticks(ticks=range(12), labels=[f'Month {m+1}' for m in range(12)])
        plt.grid(True, axis='y')
        plt.show()

# Example usage:
# Assuming df_cleaned is your DataFrame with 'state', 'month', and 'temperature'
# plot_boxplot_by_month_by_state(df_cleaned, 'temperature', 'Temperature (°C)', 'Temperature Distribution')
# plot_boxplot_by_month_by_state(df_cleaned, 'humidity', 'Humidity (%)', 'Humidity Distribution')
# plot_boxplot_by_month_by_state(df_cleaned, 'wind_speed', 'Wind Speed (m/s)', 'Wind Speed Distribution')

### 5.2.1 Temperature

#### Timeseries graph for temperature of each state

In [ ]:
# Timeseries graph for temperature of each state
plot_timeseries_by_state(df_cleaned, 'temperature', 'Temperature (°C)', 'Temperature')

#### Distribution of temperature for each state

In [ ]:
# Visualize the distribution of temperature for each state
plot_distribution_by_state(df_cleaned, 'temperature', 'Temperature (°C)', 'Distribution of Temperature')

#### Distribution of temperature for each month using box plot

In [ ]:
# Visualize the distribution of temperature for each month in each state using box plot
plot_boxplot_by_month_by_state(df_cleaned, 'temperature', 'Temperature (°C)', 'Temperature Distribution')

### 5.2.2 Pressure

#### Timeseries graph for pressure of each state

In [ ]:
# Timeseries graph for pressure of each state
plot_timeseries_by_state(df_cleaned, 'pressure', 'Pressure (hPa)', 'Pressure')

#### Distribution of pressure for each state

In [ ]:
# Visualize the distribution of pressure for each state
plot_distribution_by_state(df_cleaned, 'pressure', 'Pressure (hPa)', 'Distribution of Pressure')

#### Distribution of pressure for each month using box plot

In [ ]:
# Visualize of pressure
plot_boxplot_by_month_by_state(df_cleaned, 'pressure', 'Pressure (hPa)', 'Pressure Distribution')

### 5.2.3 Dew Point

#### Timeseries graph for dew point of each state

In [ ]:
# timeseries graph for dew_point of each state
plot_timeseries_by_state(df_cleaned, 'dew_point', 'Dew Point (°C)', 'Dew Point')

#### Distribution of dew point for each state

In [ ]:
# Distrburion of dew point for each state
plot_distribution_by_state(df_cleaned, 'dew_point', 'Dew Point (°C)', 'Distribution of Dew Point')

#### Distribution of dew point for each month using box plot

In [ ]:
# Distribution of dew point for each month using box plot
plot_boxplot_by_month_by_state(df_cleaned, 'dew_point', 'Dew Point (°C)', 'Dew Point Distribution')

### 5.2.4 Humidity

#### Timeseries graph for humidity of each state

In [ ]:
# timeseries for humidity of each state
plot_timeseries_by_state(df_cleaned, 'humidity', 'Humidity (%)', 'Humidity')

#### Distribution of humidity for each state

In [ ]:
# Distribution of humidity for each state
plot_distribution_by_state(df_cleaned, 'humidity', 'Humidity (%)', 'Distribution of Humidity')

#### Distribution of humidity for each month using box plot

In [ ]:
# Distribution of humidity for each month using box plot
plot_boxplot_by_month_by_state(df_cleaned, 'humidity', 'Humidity (%)', 'Humidity Distribution')

### 5.2.5 Wind Speed

#### Timeseries graph for wind speed of each state

In [ ]:
# timeseries graph for wind_speed of each state
plot_timeseries_by_state(df_cleaned, 'wind_speed', 'Wind Speed (m/s)', 'Wind Speed')

#### Distribution of wind speed for each state

In [ ]:
# Distribution of wind speed for each state
plot_distribution_by_state(df_cleaned, 'wind_speed', 'Wind Speed (m/s)', 'Distribution of Wind Speed')

#### Distribution of wind speed for each month using box plot

In [ ]:
# Distribution of wind speed for each month using box plot
plot_boxplot_by_month_by_state(df_cleaned, 'wind_speed', 'Wind Speed (m/s)', 'Wind Speed Distribution')

### 5.2.6 Wind Chill

#### Timeseries graph for wind chill of each state

In [ ]:
# timeseries graph for wind_chill of each state
plot_timeseries_by_state(df_cleaned, 'wind_chill', 'Wind Chill (°C)', 'Wind Chill')

#### Distribution of wind chill for each state

In [ ]:
# Distribution of wind chill for each state
plot_distribution_by_state(df_cleaned, 'wind_chill', 'Wind Chill (°C)', 'Distribution of Wind Chill')

#### Distribution of wind chill for each month using box plot

In [ ]:
# Distriburion of wind chill for each month using box plot
plot_boxplot_by_month_by_state(df_cleaned, 'wind_chill', 'Wind Chill (°C)', 'Wind Chill Distribution')

### 5.2.7 UV Index

#### Timeseries graph for uv index of each state

In [ ]:
# timeseries graph for uv_index of each state
plot_timeseries_by_state(df_cleaned, 'uv_index', 'UV Index', 'UV Index')

#### Distriburion of uv index for each state

In [ ]:
# Distribution of uv index for each state
plot_distribution_by_state(df_cleaned, 'uv_index', 'UV Index', 'Distribution of UV Index')

#### Distribution of uv index for each month using box plot

In [ ]:
# Distribution of uv index for each month using box plot
plot_boxplot_by_month_by_state(df_cleaned, 'uv_index', 'UV Index', 'UV Index Distribution')

### 5.2.8 Monthly Pattern of each Weather Variable Every State

In [ ]:
# Generate a time-series line graph for each state to visualize monthly weather patterns across multiple years, with each year's trend represented as a separate line indicator.

# Prepare data for plotting
# Ensure 'date' is datetime type and create a 'year' column
df_cleaned['date'] = pd.to_datetime(df_cleaned['date'])
df_cleaned['year'] = df_cleaned['date'].dt.year
df_cleaned['month'] = df_cleaned['date'].dt.month # Ensure month is present for aggregation

# Group data by state, year, and month to get monthly averages
monthly_avg_weather = df_cleaned.groupby(['state', 'year', 'month'])[desired_columns].mean().reset_index()

# Create a new date column representing the start of the month
monthly_avg_weather['month_start_date'] = pd.to_datetime(monthly_avg_weather[['year', 'month']].assign(day=1))

# Set the style for the plots
plt.style.use('seaborn-v0_8-whitegrid')

# Get unique states
states = monthly_avg_weather['state'].unique()

# List of weather variables to plot
weather_variables = ['temperature', 'pressure', 'dew_point', 'humidity', 'wind_speed', 'wind_chill', 'uv_index']

# Create a line plot for each weather variable, with lines for each year within each state
for state in states:
    state_df = monthly_avg_weather[monthly_avg_weather['state'] == state].copy()

    for var in weather_variables:
        if var not in state_df.columns:
            print(f"Warning: Variable '{var}' not found in data for {state}. Skipping.")
            continue

        plt.figure(figsize=(16, 6))

        # Use seaborn to plot lines for each year
        sns.lineplot(
            data=state_df,
            x='month',        # Use month as the x-axis
            y=var,
            hue='year',       # Separate lines by year
            palette='viridis', # Color palette
            marker='o',       # Add markers for data points
            legend='full'
        )

        plt.title(f'{var.replace("_", " ").title()} Trends by Year for {state}')
        plt.xlabel('Month')
        plt.ylabel(var.replace("_", " ").title())
        plt.xticks(range(1, 13), ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']) # Label months
        plt.legend(title='Year')
        plt.grid(True)
        plt.tight_layout() # Adjust layout to prevent labels overlapping
        plt.show()


# 6.0 Handle Outlier

### Function to check columns with outliers

In [ ]:
# function to check columns with outliers
def detect_outliers_iqr(df, columns):
    """
    Detects outliers in specified numerical columns using the IQR method for each state.

    Args:
        df (pd.DataFrame): The input DataFrame. Must have a 'state' column.
        columns (list): A list of column names to check for outliers.

    Returns:
        dict: A dictionary where keys are state names and values are dictionaries.
              The inner dictionary has column names as keys and a list of outlier indices as values.
    """
    outliers_dict = {}
    states = df['state'].unique()

    for state in states:
        state_df = df[df['state'] == state].copy()
        state_outliers = {}

        for col in columns:
            if col in state_df.select_dtypes(include=np.number).columns: # Only check numeric columns
                Q1 = state_df[col].quantile(0.25)
                Q3 = state_df[col].quantile(0.75)
                IQR = Q3 - Q1

                lower_bound = Q1 - 1.5 * IQR
                upper_bound = Q3 + 1.5 * IQR

                # Find indices where the value is outside the bounds
                outlier_indices = state_df[(state_df[col] < lower_bound) | (state_df[col] > upper_bound)].index.tolist()

                if outlier_indices:
                    state_outliers[col] = outlier_indices

        if state_outliers:
            outliers_dict[state] = state_outliers

    return outliers_dict

### Function to visualize the outliers in box plot

In [ ]:
# function to visualize the outliers in box plot graph for  states for each column
def plot_outliers_boxplot_by_state(df, columns, state_col='state'):
    """
    Generates box plots for specified numerical columns, grouping by state
    to visualize outliers for each column across different states on the same plot.

    Args:
        df (pd.DataFrame): The input DataFrame. Must have a 'state' column.
        columns (list): A list of numerical column names to visualize.
        state_col (str): The name of the column representing states (default is 'state').
    """
    # Set the style for the plots
    plt.style.use('seaborn-v0_8-whitegrid')

    # Ensure specified columns are numerical and exist
    numeric_cols = [col for col in columns if col in df.select_dtypes(include=np.number).columns]
    if not numeric_cols:
        print("No valid numerical columns provided for plotting.")
        return

    # Create a box plot for each numerical column
    for col in numeric_cols:
        plt.figure(figsize=(15, 8))
        sns.boxplot(x=state_col, y=col, data=df)
        plt.title(f'Box Plot of {col.replace("_", " ").title()} by State (Outlier Visualization)')
        plt.xlabel('State')
        plt.ylabel(col.replace("_", " ").title())
        plt.xticks(rotation=45, ha='right') # Rotate labels for better readability
        plt.tight_layout() # Adjust layout
        plt.show()

## 6.1 Detect the Outliers

In [ ]:
# Identify numerical columns to check for outliers
numerical_cols_to_check = ['temperature', 'pressure', 'dew_point', 'humidity', 'wind_speed', 'wind_chill', 'uv_index']

In [ ]:
# Detect outliers using IQR method per state
outliers_by_state = detect_outliers_iqr(df_cleaned, numerical_cols_to_check)

# Print the found outliers
if outliers_by_state:
    print("Outliers detected per state and column (using IQR):")
    for state, column_outliers in outliers_by_state.items():
        print(f"\n--- State: {state} ---")
        for col, indices in column_outliers.items():
            print(f"  Column '{col}': {len(indices)} outliers detected at indices: {indices[:10]}...") # Print first 10 indices as example
else:
    print("No significant outliers detected using the IQR method.")

In [ ]:
# Visualize outliers using box plots per state for each column
plot_outliers_boxplot_by_state(df_cleaned, numerical_cols_to_check, state_col='state')

## 6.2 Check Skewness

*   **Positive values**: Above-average or stronger association for that feature in that state
*   **Negative values**: Below-average or weaker association
*   **Magnitude**: Larger magnitude (further from 0) means stronger deviation or influence

In [ ]:
# Check skewness for each numerical column per state
print("\nSkewness per state and column:")
skewness_by_state = df_cleaned.groupby('state')[numerical_cols_to_check].skew()
skewness_by_state

In [ ]:
# Visualize skewness
plt.figure(figsize=(12, 8))
sns.heatmap(skewness_by_state, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Skewness of Numerical Columns by State')
plt.xlabel('Weather Variable')
plt.ylabel('State')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

**Column-wise Interpretation**

| Column        | Interpretation                                                                 |
|---------------|--------------------------------------------------------------------------------|
| `temperature` | Positive values indicate states where temperature levels are higher than average post-imputation. |
| `pressure`    | High values = states where atmospheric pressure is higher relative to others.  |
| `dew_point`   | Negative values = lower moisture or drier air compared to average.             |
| `humidity`    | Negative = lower humidity; positive = more humid.                              |
| `wind_speed`  | Values >1 (like in Johor, Kelantan) = very strong winds. Selangor is unusual with negative wind speed. |
| `wind_chill`  | Reflects cooling effect of wind. Mostly positive, meaning most states have moderate wind chill effects. |
| `uv_index`    | Positive = stronger UV exposure (e.g. Pulau Pinang); Negative = weaker UV (e.g. Perlis, Selangor). |

**State-specific Highlights**

| State              | Key Observations                                                                                                |
|--------------------|-----------------------------------------------------------------------------------------------------------------|
| Johor              | High wind speed (+1.118), typical temperature/pressure, low dew point → likely hot & windy.                       |
| Kedah / Kelantan / Labuan | Consistently high pressure & wind; low dew point = dry & pressurized conditions.                              |
| Negeri Sembilan    | Stands out with high temperature (+0.61), negative pressure, and positive humidity → possibly warm and humid, unstable pressure patterns. |
| Selangor           | Anomalously high pressure (+1.07), but negative wind speed (-0.27) → possibly due to poor coverage or random imputation artifacts. |
| Perlis             | Low UV and moderate humidity (+0.28), but generally neutral.                                                    |
| Sarawak & Sabah    | High wind speed and chill, moderately humid = tropical windy conditions.                                        |

### Suggestion for Future Improvement
Some oddities (like Selangor's negative wind speed and high pressure, Negeri Sembilan's negative pressure) suggest:

*  There may still be bias introduced by limited original data or imputation based on sparse historical patterns.

*  Consider further smoothing or imputation verification, e.g., using time-series interpolation, seasonal averages, or k-NN imputation.

## 6.3 Replace the Outliers

In [ ]:
# Function to replace outliers with lower/upper bounds using IQR method per state
def cap_outliers_iqr_by_state(df, columns):
    """
    Replaces outliers in specified numerical columns with the calculated lower/upper
    bounds using the IQR method, performed independently for each state.

    Args:
        df (pd.DataFrame): The input DataFrame. Must have a 'state' column.
        columns (list): A list of numerical column names to process.

    Returns:
        pd.DataFrame: The DataFrame with outliers capped.
    """
    df_capped = df.copy()
    states = df_capped['state'].unique()

    for state in states:
        state_indices = df_capped[df_capped['state'] == state].index
        state_df = df_capped.loc[state_indices].copy() # Get a writable copy for the state

        for col in columns:
            if col in state_df.select_dtypes(include=np.number).columns:
                Q1 = state_df[col].quantile(0.25)
                Q3 = state_df[col].quantile(0.75)
                IQR = Q3 - Q1

                lower_bound = Q1 - 1.5 * IQR
                upper_bound = Q3 + 1.5 * IQR

                # Apply capping directly to the slice corresponding to the state
                df_capped.loc[state_indices, col] = df_capped.loc[state_indices, col].apply(
                    lambda x: lower_bound if x < lower_bound else (upper_bound if x > upper_bound else x)
                )

    return df_capped


In [ ]:
# Apply the outlier capping function
df_cleaned_capped = cap_outliers_iqr_by_state(df_cleaned.copy(), numerical_cols_to_check)

# Verify by checking outliers again (should be fewer or none)
outliers_after_capping = detect_outliers_iqr(df_cleaned_capped, numerical_cols_to_check)

if outliers_after_capping:
    print("\nOutliers detected after capping (using IQR):")
    for state, column_outliers in outliers_after_capping.items():
         print(f"\n--- State: {state} ---")
         for col, indices in column_outliers.items():
             print(f"  Column '{col}': {len(indices)} outliers detected at indices: {indices[:10]}...")
else:
    print("\n✅ No significant outliers detected after capping using the IQR method.")

In [ ]:
plot_outliers_boxplot_by_state(df_cleaned_capped, numerical_cols_to_check, state_col='state')

# 7.0 Export the Processed Weather Dataset

In [ ]:
df_processed = df_cleaned_capped.copy()
df_processed.head()

Remove date column

In [ ]:
# remove date column
df_processed = df_processed.drop(columns=['date'])
df_processed.head()

Check on dataset info

In [ ]:
df_processed.info()

Check on size of dataset and missing statistics

In [ ]:
print(f"Size of the final processed dataset: {df_processed.shape}\n")
print(missing_statistics(df_processed))

Export the dataset into Drive

In [ ]:
# export the dataset into drive
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Define the path to save the file in your Google Drive
# You can change 'My Drive' to the name of your Drive folder if it's different
# And change the filename 'processed_weather_data.csv' if needed
export_path = '/content/drive/My Drive/FYP_Fish/weather_data/processed_weather_data.csv'

# Export the DataFrame to a CSV file
df_processed.to_csv(export_path, index=False)

print(f"Dataset successfully exported to: {export_path}")